In [ ]:
# Paths come from the memtune package (see docs/INSTALL.md): no machine-specific locations.
#   data   -> data/lbfgs_runs.csv   (override: MEMTUNE_DATA_DIR)
#   models -> models/              (override: MEMTUNE_MODELS_DIR)
#   output -> results/             (override: MEMTUNE_RESULTS_DIR)
from memtune import paths
(paths.results_dir() / "retrained_models").mkdir(parents=True, exist_ok=True)  # notebooks never overwrite models/


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, LinearRegression
import xgboost as xgb
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 20)
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
import joblib
import csv

### Data Preprocessing

In [ ]:
df = pd.read_csv(str(paths.dataset_path()))
first = ["name", "nvar"]
df = df[first + [c for c in df.columns if c not in first]]
df["nvar"].value_counts
df = df.sort_values(["name", "nvar", "mem"]).reset_index()
df

### Identify the features and targets for this model

In [ ]:
feature_cols_reg = [
                    # Core Features
                    "nvar", 
                    "mem",
                    "tree_length", 
                    "tree_depth", 
                    "time_obj",  # initial eval
                    "time_grad",
                    "extract_duration_problem",
                    
                    # Expression Tree Features
                    "highest_degree (ExprTree)",
                    "count_plus_minus",
                    "count_trigonometric_function",
                    "count_exponential_function",
                    "count_op_function",

                    # allocation
                    "alloc_obj",
                    "alloc_grad",
                    "total_alloc",
                  ]

target_cols_reg = [ "neval_obj",
                    "neval_grad",
                    "timed_bytes"] #(neval_obj, neval_grad, timed_bytes)

target_col_model = ["stats_elapsed_time"]

group_key = ["name", "nvar"]
all_cols = list(set(feature_cols_reg + target_cols_reg + target_col_model + group_key))
solver_metrics = ['neval_grad', 'neval_obj', 'timed_bytes']
df[solver_metrics] = df[solver_metrics].fillna(0)
df = df[all_cols]
df.isna().sum().sort_values(ascending=False)

In [ ]:
df = df.copy()
df["mem_count"] = df.groupby(["name", "nvar"]).transform('size')
df = df[df["mem_count"] == 100].copy().reset_index(drop=True)
df

In [ ]:
# unique instances: one row per (problem, nvar)
instances = df[group_key].drop_duplicates()

# shuffle instances
instances = instances.sample(frac=1, random_state=66).reset_index(drop=True)

n = len(instances)
n_train = int(0.8 * n)

train_inst = instances.iloc[:n_train]
test_inst  = instances.iloc[n_train:]

# assign rows to splits by (problem, nvar)
train_df = df.merge(train_inst, on=["nvar", "name"], how="inner").reset_index(drop=True)
test_df  = df.merge(test_inst,  on=["nvar", "name"], how="inner").reset_index(drop=True)
all_df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)
all_df

In [ ]:
X_train = train_df[feature_cols_reg].to_numpy(dtype=float)
X_test  = test_df[feature_cols_reg].to_numpy(dtype=float)

y_train = np.log1p(train_df[target_cols_reg].to_numpy(dtype=float))
y_test  = np.log1p(test_df[target_cols_reg].to_numpy(dtype=float))

X_all = np.concatenate([X_train, X_test], axis=0)

### Global Performance Ratio Function

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import gmean

def evaluate_model(model, label="Model"):
    """
    Evaluates a stage-2 model using global X_test_time and global test_df.
    Includes arithmetic mean, geometric mean, and maximum performance ratio metrics.
    """
    predictions = model.predict(X_test_time)
    working_df = test_df.copy() 
    
    working_df["predicted_time"] = predictions
    group_cols = ["name", "nvar"]

    # 1. Model Selection
    model_choices = (
        working_df.sort_values(by=group_cols + ["predicted_time", "mem"], ascending=[True, True, True, False])
        .groupby(group_cols).first().reset_index()
        [group_cols + ["mem", "predicted_time", "stats_elapsed_time"]]
        .rename(columns={"mem": "predicted_mem", "stats_elapsed_time": "actual_time"})
    )

    # 2. Oracle Selection
    oracle_choices = (
        working_df.sort_values(by=group_cols + ["stats_elapsed_time", "mem"], ascending=[True, True, True, False])
        .groupby(group_cols).first().reset_index()
        [group_cols + ["mem", "stats_elapsed_time"]]
        .rename(columns={"mem": "best_mem", "stats_elapsed_time": "best_time"})
    )

    # 3. Merge Matrix & Apply Hardened 0.10s Absolute Tolerance Filter
    final_df = pd.merge(model_choices, oracle_choices, on=group_cols).copy()
    final_df['actual_time'] = pd.to_numeric(final_df['actual_time'], errors='coerce')
    final_df['best_time'] = pd.to_numeric(final_df['best_time'], errors='coerce')

    final_df['raw_ratio'] = final_df['actual_time'] / final_df['best_time']
    time_diff = np.abs(final_df['actual_time'] - final_df['best_time'])
    final_df['performance_ratio'] = np.where(time_diff <= 0.10, 1.0, final_df['raw_ratio'])

    # Reorder display columns
    column_order = group_cols + ["predicted_mem", "predicted_time", "actual_time", "best_mem", "best_time", "performance_ratio", "raw_ratio"]
    final_df = final_df[column_order]

    # Output Summary Blocks
    print("\n" + "="*145)
    print(f"PER-PROBLEM PERFORMANCE PROFILE SUMMARY ({label.upper()})")
    print("="*145)
    print(final_df.to_string(index=False, formatters={
        'predicted_time': '{:.4f}'.format, 'actual_time': '{:.4f}'.format,
        'best_time': '{:.4f}'.format, 'performance_ratio': '{:.4f}'.format, 'raw_ratio': '{:.4f}'.format
    }))
    print("="*145)

    total_actual = final_df['actual_time'].sum()
    total_best = final_df['best_time'].sum()
    
    # Calculate Core Evaluation Statistics
    global_mean_ratio = final_df['performance_ratio'].mean()
    geom_mean_ratio = gmean(final_df['performance_ratio'])  # Clean Scipy implementation
    max_performance_ratio = final_df['performance_ratio'].max()
    
    print("\n" + "="*50)
    print(f"AGGREGATE METRICS ({label.upper()})")
    print("="*50)
    print(final_df['performance_ratio'].quantile([0.50, 0.75, 0.90]).to_string(float_format='{:.4f}'.format))
    print("-"*50)
    print(f"Global Mean Perf. Ratio     : {global_mean_ratio:.4f}")
    print(f"Geometric Mean Perf. Ratio  : {geom_mean_ratio:.4f}")
    print(f"Maximum Performance Ratio   : {max_performance_ratio:.4f}")
    print("-"*50)
    print(f"Total Actual Compute Time   : {total_actual:.4f}s")
    print(f"Total Oracle Optimal Time   : {total_best:.4f}s")
    print("="*50)
    
    return final_df

### Model Selection and Hyperparameter Tuning

### Stage 1: XGBoost (native multi-output) — predicting neval_obj, neval_grad, timed_bytes

`XGBRegressor` handles multi-output natively when `y` has shape `(n, k)` — no wrapper needed.

In [ ]:
# # Hyperparameter search for Stage 1 XGBoost
# # Uncomment to run

# param_distributions = {
#     'n_estimators':      randint(50, 500),
#     'learning_rate':     uniform(0.01, 0.29),      # [0.01, 0.30]
#     'max_depth':         randint(2, 10),
#     'subsample':         uniform(0.5, 0.5),         # [0.5, 1.0]
#     'colsample_bytree':  uniform(0.4, 0.6),         # [0.4, 1.0]
#     'min_child_weight':  randint(1, 20),
#     'gamma':             uniform(0, 0.5),
#     'reg_alpha':         uniform(0, 1.0),
#     'reg_lambda':        uniform(0.5, 2.0),
# }

# xgb_model = xgb.XGBRegressor(
#     tree_method='hist',
#     random_state=42,
#     n_jobs=1
# )

# random_search = RandomizedSearchCV(
#     xgb_model,
#     param_distributions=param_distributions,
#     n_iter=200,
#     cv=5,
#     scoring='neg_mean_squared_error',
#     n_jobs=4,
#     random_state=42,
#     verbose=3
# )

# random_search.fit(X_train, y_train)

# print("\n" + "="*80)
# print("BEST RESULTS")
# print("="*80)
# print("Best parameters:", random_search.best_params_)
# print("Best CV score (neg MSE):", random_search.best_score_)
# print("Actual MSE:", -random_search.best_score_)

# results_df = pd.DataFrame(random_search.cv_results_)
# results_df['actual_mse'] = -results_df['mean_test_score']
# results_df = results_df.sort_values('actual_mse')

# top_cols = ['mean_test_score', 'std_test_score', 'mean_fit_time',
#             'param_n_estimators', 'param_learning_rate', 'param_max_depth',
#             'param_subsample', 'param_colsample_bytree', 'param_min_child_weight']
# print(results_df[top_cols].head(20).to_string())

# results_df.to_csv('xgb_randomsearch_stage1_results.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt

best_model = joblib.load(str(paths.models_dir() / "stage1/xgboost.pkl"))
# best_model = random_search.best_estimator_

xgb_importances = best_model.feature_importances_
indices_xgb = np.argsort(xgb_importances)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(range(len(feature_cols_reg)), xgb_importances[indices_xgb])
plt.xticks(range(len(feature_cols_reg)), [feature_cols_reg[i] for i in indices_xgb], rotation=45, ha="right")
plt.ylabel("Feature importance")
plt.title("XGBoost feature importances (Stage 1)")
plt.tight_layout()
plt.show()

# joblib.dump(best_model, str(paths.results_dir() / "retrained_models" / "best_xgb_model_reg.pkl"))

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

pred = best_model.predict(X_test)

# multioutput='raw_values' returns an array of metrics, one per target
test_mse_per_target = mean_squared_error(y_test, pred, multioutput='raw_values')
test_r2_per_target  = r2_score(y_test, pred, multioutput='raw_values')

# Loop through each target column to print its specific MSE and R²
for i, col in enumerate(target_cols_reg):
    print(f"\nTarget: {col}")
    print(f"  MSE: {test_mse_per_target[i]:.4f}")
    print(f"  R²:  {test_r2_per_target[i]:.4f}")

# Quantiled Relative Error Distribution
relative_error = np.abs(pred - y_test) / (np.abs(y_test) + 1e-8)

quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
for i, col in enumerate(target_cols_reg):
    print(f"\n{col}:")
    print(f"Quantile (q) | Relative Error Value Q(q)")
    
    # Compute the error magnitudes at each specific quantile marker
    q_values = np.quantile(relative_error[:, i], quantiles)
    
    for q, val in zip(quantiles, q_values):
        print(f"  Q({q:.2f}):  {val:.4f}")

### Stage 2: Using the **predicted** values of ["neval_obj", "neval_grad", "timed_bytes"] to predict time

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_predict

# 1. Define your log-space targets for Stage 2 (Actual Time)
y_train_time = np.log1p(train_df[target_col_model].to_numpy(dtype=float)).ravel()
y_test_time  = np.log1p(test_df[target_col_model].to_numpy(dtype=float)).ravel()

# 2. CROSS-VALIDATION: Generate realistic, noisy 3-column features for Stage 2
X_train_time = pd.DataFrame(
    cross_val_predict(best_model, X_train, y_train, cv=5, n_jobs=-1),
    columns=target_cols_reg
)

# 3. Fit the Stage 1 model on the full training set to learn the 3 metrics
best_model.fit(X_train, y_train)

# 4. Generate standard 3-column test features for Stage 2
X_test_time = pd.DataFrame(
    best_model.predict(X_test),
    columns=target_cols_reg
)

In [ ]:
# # Hyperparameter search for Stage 2 XGBoost (predicted inputs)
# # Uncomment to run

# param_distributions = {
#     'n_estimators':      randint(50, 500),
#     'learning_rate':     uniform(0.01, 0.29),
#     'max_depth':         randint(2, 10),
#     'subsample':         uniform(0.5, 0.5),
#     'colsample_bytree':  uniform(0.4, 0.6),
#     'min_child_weight':  randint(1, 20),
#     'gamma':             uniform(0, 0.5),
#     'reg_alpha':         uniform(0, 1.0),
#     'reg_lambda':        uniform(0.5, 2.0),
# }

# xgb_model = xgb.XGBRegressor(tree_method='hist', random_state=42, n_jobs=1)

# time_random_search = RandomizedSearchCV(
#     xgb_model,
#     param_distributions=param_distributions,
#     n_iter=200,
#     cv=5,
#     scoring='neg_mean_squared_error',
#     n_jobs=4,
#     random_state=42,
#     verbose=3,
#     return_train_score=True,
#     error_score='raise'
# )

# time_random_search.fit(X_train_time, y_train_time)

# print("\n" + "="*80)
# print("BEST RESULTS")
# print("="*80)
# print("Best parameters:", time_random_search.best_params_)
# print("Best CV score (neg MSE):", time_random_search.best_score_)
# print("Actual MSE:", -time_random_search.best_score_)

# results_df = pd.DataFrame(time_random_search.cv_results_)
# results_df['actual_mse'] = -results_df['mean_test_score']
# results_df = results_df.sort_values('actual_mse')

# top_cols = ['mean_test_score', 'std_test_score', 'mean_fit_time',
#             'param_n_estimators', 'param_learning_rate', 'param_max_depth',
#             'param_subsample', 'param_colsample_bytree', 'param_min_child_weight']
# print(results_df[top_cols].head(20).to_string())

# results_df.to_csv('xgb_randomsearch_stage2_pred_results.csv', index=False)

In [ ]:
time_best_model = joblib.load(str(paths.models_dir() / "legacy/stage2_xgb_on_predicted_stage1.pkl"))
# time_best_model = time_random_search.best_estimator_
accuracy = time_best_model.score(X_test_time, y_test_time)
print(f"Time prediction accuracy based on predicted values={accuracy}")

# joblib.dump(time_best_model, str(paths.results_dir() / "retrained_models" / "best_xgb_model_time.pkl"))

In [ ]:
pred = time_best_model.predict(X_test_time)
test_mse = mean_squared_error(y_test_time, pred)
test_r2  = r2_score(y_test_time, pred)
print(f"MSE={test_mse:.4f}  R²={test_r2:.4f}")

# Compute pointwise relative error matching the paper's formula: |y - y_hat| / (|y| + delta)
relative_error = np.abs(pred - y_test_time) / (np.abs(y_test_time) + 1e-8)

# Quantiled Relative Error Distribution
quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
for i, col in enumerate(target_col_model):
    print(f"\n{col}:")
    print(f"Quantile (q) | Relative Error Value Q(q)")
    
    # Safe extraction whether relative_error is a 1D vector or a 2D column matrix
    error_data = relative_error[:, i] if relative_error.ndim > 1 else relative_error
    q_values = np.quantile(error_data, quantiles)
    
    for q, val in zip(quantiles, q_values):
        print(f"  Q({q:.2f}):  {val:.4f}")

In [ ]:
evaluate_model(joblib.load(str(paths.models_dir() / "legacy/stage2_xgb_on_predicted_stage1.pkl")))

In [ ]:
xgb_importances_time = time_best_model.feature_importances_
indices_xgb = np.argsort(xgb_importances_time)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(range(len(target_cols_reg)), xgb_importances_time[indices_xgb])
plt.xticks(range(len(target_cols_reg)), [target_cols_reg[i] for i in indices_xgb], rotation=45, ha="right")
plt.ylabel("Feature importance")
plt.title("XGBoost feature importances (Stage 2)")
plt.tight_layout()
plt.show()

### Stage 2 (variant): Using the **actual** values of ["neval_obj", "neval_grad", "timed_bytes"] to predict time

In [ ]:
X_train_time = train_df[target_cols_reg]
X_test_time  = test_df[target_cols_reg]
y_train_time = np.log1p(train_df[target_col_model].to_numpy(dtype=float)).ravel()
y_test_time  = np.log1p(test_df[target_col_model].to_numpy(dtype=float)).ravel()

In [ ]:
# Quick sanity check with default XGBoost
time_predictor = xgb.XGBRegressor(tree_method='hist', random_state=42)
time_predictor.fit(X_train_time, y_train_time)
time_predictor.score(X_test_time, y_test_time)

In [ ]:
# # Hyperparameter search for Stage 2 XGBoost (raw inputs)
# # Uncomment to run

# param_distributions = {
#     'n_estimators':      randint(50, 500),
#     'learning_rate':     uniform(0.01, 0.29),
#     'max_depth':         randint(2, 10),
#     'subsample':         uniform(0.5, 0.5),
#     'colsample_bytree':  uniform(0.4, 0.6),
#     'min_child_weight':  randint(1, 20),
#     'gamma':             uniform(0, 0.5),
#     'reg_alpha':         uniform(0, 1.0),
#     'reg_lambda':        uniform(0.5, 2.0),
# }

# xgb_model = xgb.XGBRegressor(tree_method='hist', random_state=42, n_jobs=1)

# time_random_search_raw = RandomizedSearchCV(
#     xgb_model,
#     param_distributions=param_distributions,
#     n_iter=200,
#     cv=5,
#     scoring='neg_mean_squared_error',
#     n_jobs=6,
#     random_state=42,
#     verbose=2,
#     return_train_score=True,
#     error_score='raise'
# )

# time_random_search_raw.fit(X_train_time, y_train_time)

# print("\n" + "="*80)
# print("BEST RESULTS")
# print("="*80)
# print("Best parameters:", time_random_search_raw.best_params_)
# print("Best CV score (neg MSE):", time_random_search_raw.best_score_)
# print("Actual MSE:", -time_random_search_raw.best_score_)

# results_df = pd.DataFrame(time_random_search_raw.cv_results_)
# results_df['actual_mse'] = -results_df['mean_test_score']
# results_df = results_df.sort_values('actual_mse')

# top_cols = ['mean_test_score', 'std_test_score', 'mean_fit_time',
#             'param_n_estimators', 'param_learning_rate', 'param_max_depth',
#             'param_subsample', 'param_colsample_bytree', 'param_min_child_weight']
# print(results_df[top_cols].head(20).to_string())

# results_df.to_csv('xgb_randomsearch_stage2_raw_results.csv', index=False)

In [ ]:
time_best_model_raw = joblib.load(str(paths.models_dir() / "stage2_ground_truth/xgboost.pkl"))
# time_best_model_raw = time_random_search_raw.best_estimator_
accuracy = time_best_model_raw.score(X_test_time, y_test_time)
print(f"Time prediction accuracy based on raw values={accuracy}")

# joblib.dump(time_best_model_raw, str(paths.results_dir() / "retrained_models" / "best_xgb_model_time_raw.pkl"))

In [ ]:
pred = time_best_model_raw.predict(X_test_time)
test_mse = mean_squared_error(y_test_time, pred)
test_r2  = r2_score(y_test_time, pred)
print(f"MSE={test_mse:.4f}  R²={test_r2:.4f}")

# Compute pointwise relative error matching the paper's formula: |y - y_hat| / (|y| + delta)
relative_error = np.abs(pred - y_test_time) / (np.abs(y_test_time) + 1e-8)

# Quantiled Relative Error Distribution
quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
for i, col in enumerate(target_col_model):
    print(f"\n{col}:")
    print(f"Quantile (q) | Relative Error Value Q(q)")
    
    # Safe extraction whether relative_error is a 1D vector or a 2D column matrix
    error_data = relative_error[:, i] if relative_error.ndim > 1 else relative_error
    q_values = np.quantile(error_data, quantiles)
    
    for q, val in zip(quantiles, q_values):
        print(f"  Q({q:.2f}):  {val:.4f}")

In [ ]:
evaluate_model(joblib.load(str(paths.models_dir() / "stage2_ground_truth/xgboost.pkl")))

In [ ]:
xgb_importances_raw = time_best_model_raw.feature_importances_
indices_xgb = np.argsort(xgb_importances_raw)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(range(len(target_cols_reg)), xgb_importances_raw[indices_xgb])
plt.xticks(range(len(target_cols_reg)), [target_cols_reg[i] for i in indices_xgb], rotation=45, ha="right")
plt.ylabel("Feature importance")
plt.title("Stage 2 Feature Importances")
plt.tight_layout()
plt.show()